# imports

In [1]:
# !git clone https://github.com/moryacho/street-pattern-classifier.git

# !pip install huggingface_hub geopandas osmnx
# !pip install torch_geometric
# !pip install "folium>=0.12" matplotlib mapclassify

# !pip uninstall torch torchvision torchaudio -y
# !pip cache purge
# !pip install torch torchvision torchaudio

import torch

In [2]:
import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
import time
from tqdm import tqdm
import os
import networkx as nx
import json

import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats
import warnings

from shapely.geometry import LineString, Polygon, Point
import shapely
import pickle
import ast



# cities 

## names

In [10]:
osm_city_names = [
    # East Asia
    "Beijing, China",
    "Chengdu, China",
    "Chongqing, China",
    "Guangzhou, China",
    "Hangzhou, China",
    "Shanghai, China",
    "Shenzhen, China",
    "Wuhan, China",
    "Nanjing, China",
    "Taipei, Taiwan",
    # "Tokyo, Japan", # multyoplygon: so buffer in the ocean
    "Osaka, Japan",
    "Seoul, South Korea",
    
    # South Asia
    "Bangkok, Thailand",
    "Delhi, India",
    "Singapore, Singapore",
    "Kuala Lumpur, Malaysia",
    # "Mumbai, India",
    "Hyderabad, India",
    "Kathmandu, Nepal",
    
    # Oceania
    "Melbourne, Australia",
    "Sydney, Australia",
    "Auckland, New Zealand",
    "Brisbane, Australia",
    
    # Africa
    "Cairo, Egypt",
    "Casablanca, Morocco",
    "Lagos, Nigeria",
    "Nairobi, Kenya",
    # "Johannesburg, South Africa",
    
    # Europe
    "Amsterdam, Netherlands",
    "Brussels, Belgium",
    "London, United Kingdom",
    "Paris, France",
    "Berlin, Germany",
    "Munich, Germany",
    "Madrid, Spain",
    "Milan, Italy",
    "Moscow, Russia",
    "Stockholm, Sweden",
    "Bucharest, Romania",
    "Saint Petersburg, Russia",
    
    # North America
    "New York City, USA",  # Важно: именно "New York City", а не просто "New York"
    "Los Angeles, USA",
    "Detroit, USA",
    "Philadelphia, USA",
    "San Antonio, USA",
    "San Diego, USA",
    "Toronto, Canada",
    "Montreal, Canada",
    "Chicago, USA",
    "Houston, USA",
    "Phoenix, USA",
    
    # South America
    "Lima, Peru",
    "Bogota, Colombia",
    "Sao Paulo, Brazil",  # Можно также "São Paulo, Brazil"
    "Caracas, Venezuela",
    "Brasilia, Brazil",    # Можно также "Brasília, Brazil"
    "Asuncion, Paraguay"
]

## centroids and buffers

In [11]:
BUFFER = 20000
data = []

for city in tqdm(osm_city_names):
    gdf = ox.geocode_to_gdf(city)
    if gdf.crs != 4326:
        print(f"Исходная CRS: {gdf.crs}")
    gdf_projected = gdf.to_crs(gdf.estimate_utm_crs())
    center_point_projected = gdf_projected.centroid.iloc[0]
    
    point_gdf_utm = gpd.GeoDataFrame(geometry=[center_point_projected], crs=gdf_projected.crs)
    buffer_utm = point_gdf_utm.buffer(BUFFER)
    buffer_geo = buffer_utm.to_crs(gdf.crs)    
    center_point_geo = gpd.GeoDataFrame(geometry=[center_point_projected], crs=gdf_projected.crs).to_crs(gdf.crs).geometry.iloc[0]
    
    data.append({
        "city_name": city,
        "centroid": center_point_geo,
        "geometry": buffer_geo.geometry.iloc[0]
    })

  0%|          | 0/55 [00:00<?, ?it/s]

100%|██████████| 55/55 [00:05<00:00, 10.91it/s]


In [12]:
gdf_centroids_polygons = gpd.GeoDataFrame(data, crs='EPSG:4326', geometry='geometry')

In [13]:
# gdf_only_centroids = gpd.GeoDataFrame(data, crs='EPSG:4326', geometry='centroid')

In [14]:
gdf_centroids_polygons.explore()

## drive graph data

In [15]:
# Source - https://stackoverflow.com/a/77965352
# Posted by Nick ODell
# Retrieved 2026-03-19, License - CC BY-SA 4.0

import osmnx as ox

# Must be an overpass instance which supports attic
ox.settings.overpass_endpoint = "https://overpass-api.de/api"

def query_year(coordinate, year):
    date = f'{year}-01-01T00:00:00Z'
    # Request attic data
    ox.settings.overpass_settings = '[out:json][timeout:{timeout}][date:"' + date + '"]{maxsize}'
    graph = ox.graph.graph_from_polygon(coordinate)
    # Restore old settings
    ox.settings.overpass_settings = '[out:json][timeout:{timeout}]{maxsize}'
    return graph


In [ ]:
# gdf_centroids_polygons['graph_2026'] = None
# gdf_centroids_polygons['graph_2020'] = None


for i in tqdm(range(len(gdf_centroids_polygons))):
    row = gdf_centroids_polygons.iloc[i]
    polygon = row['geometry']
    if gdf_centroids_polygons.at[row.name, 'graph_2026']:
        print(f"place {row.city_name} already downloaded")
        continue
    try:
        drive_graph_2026 = ox.graph_from_polygon(polygon, network_type='drive')
        drive_graph_2020 = query_year(polygon, 2020)
        
        gdf_centroids_polygons.at[row.name, 'graph_2026'] = drive_graph_2026
        gdf_centroids_polygons.at[row.name, 'graph_2020'] = drive_graph_2020
    except Exception  as e:
        print(f"place: {row.city_name} error: {e}")

    if i % 10 == 0:
        gdf_centroids_polygons.to_pickle(f"graphs_through_years//gdf_{i // 10}.pkl")

gdf_centroids_polygons.to_pickle(f"graphs_through_years//gdf_final.pkl")


 73%|███████▎  | 40/55 [3:57:06<1:28:55, 355.67s/it]


MemoryError: 

In [3]:
# gdf_centroids_polygons.to_pickle("gdf_56_cities_centroids_polygons_graphs.pkl")
gdf_centroids_polygons = gpd.GeoDataFrame(pd.read_pickle("graphs_through_years//gdf_2.pkl"), geometry="geometry")


In [12]:
gdf_centroids_polygons

,city_name,centroid,geometry,graph_2026,graph_2020
0,"Beijing, China",POINT (116.41898390046961 40.17826035972572),"POLYGON ((116.65389 40.1792, 116.65285 40.1615...","(36380695, 36380698, 36380699, 36380700, 72855...","(36380695, 36380700, 72855718, 72855723, 72855..."
1,"Chengdu, China",POINT (103.9299131329643 30.653447458925932),"POLYGON ((104.13863 30.655, 104.13778 30.6373,...","(276556859, 276556860, 276556885, 276556900, 2...","(276556799, 276556900, 276556908, 276557064, 2..."
2,"Chongqing, China",POINT (107.8658844454987 30.05462134902427),"POLYGON ((108.07308 30.04994, 108.07154 30.032...","(4663024480, 4663024482, 4663024492, 466302449...","(5510660710, 6201942791, 6201942804, 620194280..."
3,"Guangzhou, China",POINT (113.53992658306854 23.327760364828965),"POLYGON ((113.73535 23.32447, 113.73405 23.306...","(370231276, 370231305, 370231438, 445778214, 4...","(370231276, 370231305, 370231438, 445778214, 4..."
4,"Hangzhou, China",POINT (119.4686666897235 29.900427525160346),"POLYGON ((119.67561 29.89639, 119.67414 29.878...","(672831391, 672831767, 672831850, 672831851, 6...","(672831391, 672831511, 672831588, 672831767, 6..."
5,"Shanghai, China",POINT (121.84949810718098 31.245960612803987),"POLYGON ((122.0595 31.24767, 122.05867 31.2299...","(77997015, 77997031, 77998471, 77998473, 77999...","(77998471, 77998473, 77999142, 77999143, 77999..."
6,"Shenzhen, China",POINT (114.5791688697219 22.500548831919602),"POLYGON ((114.77346 22.50335, 114.77281 22.485...","(653128103, 653128189, 670742698, 685056597, 6...","(653127753, 653128103, 653128189, 670742698, 6..."
7,"Wuhan, China",POINT (114.34247757358415 30.622391630613123),"POLYGON ((114.55095 30.62649, 114.5504 30.6087...","(267620065, 286073041, 286073322, 286073497, 2...","(267620078, 286073041, 286073313, 286073322, 2..."
8,"Nanjing, China",POINT (118.84308111020037 31.926202065248283),"POLYGON ((119.05453 31.92296, 119.05311 31.905...","(29571639, 29571641, 304815373, 304815377, 304...","(29571632, 29571641, 29571710, 32918369, 30481..."
9,"Taipei, Taiwan",POINT (120.67220079121316 23.50888347070445),"POLYGON ((120.86767 23.50415, 120.86621 23.486...","(59943316, 59943336, 59943358, 59943393, 59943...","(59943316, 59943336, 59943358, 59943393, 59943..."


In [4]:
gdf = gdf_centroids_polygons.dropna(subset="graph_2026")

In [5]:
gdf

,city_name,centroid,geometry,graph_2026,graph_2020
0,"Beijing, China",POINT (116.41898390046961 40.17826035972572),"POLYGON ((116.65389 40.1792, 116.65285 40.1615...","(36380695, 36380698, 36380699, 36380700, 72855...","(36380695, 36380700, 72855718, 72855723, 72855..."
1,"Chengdu, China",POINT (103.9299131329643 30.653447458925932),"POLYGON ((104.13863 30.655, 104.13778 30.6373,...","(276556859, 276556860, 276556885, 276556900, 2...","(276556799, 276556900, 276556908, 276557064, 2..."
2,"Chongqing, China",POINT (107.8658844454987 30.05462134902427),"POLYGON ((108.07308 30.04994, 108.07154 30.032...","(4663024480, 4663024482, 4663024492, 466302449...","(5510660710, 6201942791, 6201942804, 620194280..."
3,"Guangzhou, China",POINT (113.53992658306854 23.327760364828965),"POLYGON ((113.73535 23.32447, 113.73405 23.306...","(370231276, 370231305, 370231438, 445778214, 4...","(370231276, 370231305, 370231438, 445778214, 4..."
4,"Hangzhou, China",POINT (119.4686666897235 29.900427525160346),"POLYGON ((119.67561 29.89639, 119.67414 29.878...","(672831391, 672831767, 672831850, 672831851, 6...","(672831391, 672831511, 672831588, 672831767, 6..."
5,"Shanghai, China",POINT (121.84949810718098 31.245960612803987),"POLYGON ((122.0595 31.24767, 122.05867 31.2299...","(77997015, 77997031, 77998471, 77998473, 77999...","(77998471, 77998473, 77999142, 77999143, 77999..."
6,"Shenzhen, China",POINT (114.5791688697219 22.500548831919602),"POLYGON ((114.77346 22.50335, 114.77281 22.485...","(653128103, 653128189, 670742698, 685056597, 6...","(653127753, 653128103, 653128189, 670742698, 6..."
7,"Wuhan, China",POINT (114.34247757358415 30.622391630613123),"POLYGON ((114.55095 30.62649, 114.5504 30.6087...","(267620065, 286073041, 286073322, 286073497, 2...","(267620078, 286073041, 286073313, 286073322, 2..."
8,"Nanjing, China",POINT (118.84308111020037 31.926202065248283),"POLYGON ((119.05453 31.92296, 119.05311 31.905...","(29571639, 29571641, 304815373, 304815377, 304...","(29571632, 29571641, 29571710, 32918369, 30481..."
9,"Taipei, Taiwan",POINT (120.67220079121316 23.50888347070445),"POLYGON ((120.86767 23.50415, 120.86621 23.486...","(59943316, 59943336, 59943358, 59943393, 59943...","(59943316, 59943336, 59943358, 59943393, 59943..."


# classify

In [ ]:
import sys
sys.path.insert(0, './street-pattern-classifier')
from huggingface_hub import hf_hub_download
import torch
from splits import split_graph
from block_dataset import BlockDataset
from classification import classify_blocks
from collections import Counter
from model import class_names


model_path = hf_hub_download(
        repo_id="nochka/street-pattern-classifier",
        filename="best_model.pth",
        local_dir="./models"
)
def classify_city(G, model_path):
    # subgraphs = split_graph(G, resolution=50)
    subgraphs = split_graph(G, grid_step=1000)
    dataset = BlockDataset(subgraphs)
    predictions_blocks, probabilities_blocks = classify_blocks(
        dataset,
        model_path=model_path,
        device='cpu'
    )
    predictions_list = list(predictions_blocks.values())
    class_counts = Counter(predictions_list)
    most_common_class = class_counts.most_common(1)[0][0]

    

    return class_names[most_common_class]


In [5]:
import sys
sys.path.insert(0, './street-pattern-classifier')
from huggingface_hub import hf_hub_download
import torch
from splits import split_graph
from block_dataset import BlockDataset
from classification import classify_blocks
from collections import Counter
from model import class_names
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon
from tqdm import tqdm

model_path = hf_hub_download(
    repo_id="nochka/street-pattern-classifier",
    filename="best_model.pth",
    local_dir="./models"
)

def classify_city(G, model_path, year=None):

    subgraphs = split_graph(G, grid_step=1000)
    dataset = BlockDataset(subgraphs)
    predictions_blocks, probabilities_blocks = classify_blocks(
        dataset,
        model_path=model_path,
        device='cpu'
    )
    
    # predictions_blocks - словарь {cell_id: predicted_class}
    # probabilities_blocks - словарь {cell_id: probabilities_array}
    
    # Создаем список для хранения данных
    data = []
    
    # Проходим по всем блокам в subgraphs
    for cell_id, cell_data in subgraphs.items():
        if cell_id in predictions_blocks:
            predicted_class = predictions_blocks[cell_id]
            probabilities = probabilities_blocks[cell_id]
            
            # Создаем словарь с вероятностями для каждого класса
            probs_dict = {}
            for idx, prob in enumerate(probabilities):
                class_name = class_names[idx] if idx < len(class_names) else f"class_{idx}"
                probs_dict[class_name] = prob
            
            # Добавляем данные
            data.append({
                'subgraph': cell_data['graph'],
                'polygon': cell_data['polygon'],
                'predicted_class': class_names[predicted_class] if predicted_class < len(class_names) else predicted_class,
                'predicted_class_id': predicted_class,
                'probabilities': probs_dict,
                'year': year
            })
    
    # Создаем GeoDataFrame
    gdf_results = gpd.GeoDataFrame(data, geometry='polygon', crs=G.graph.get('crs', 'EPSG:4326'))
    
    # Добавляем отдельные колонки для вероятностей каждого класса
    for class_name in class_names:
        gdf_results[f'prob_{class_name}'] = gdf_results['probabilities'].apply(lambda x: x.get(class_name, 0))
    
    return gdf_results



d:\выбирай итмо и не  выбирай вообще\urban\corr_transport_street_pattern\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
gdf_classes = gdf.copy()["city_name"]

In [8]:
gdf_classes

0             Beijing, China
1             Chengdu, China
2           Chongqing, China
3           Guangzhou, China
4            Hangzhou, China
5            Shanghai, China
6            Shenzhen, China
7               Wuhan, China
8             Nanjing, China
9             Taipei, Taiwan
10              Osaka, Japan
11        Seoul, South Korea
12         Bangkok, Thailand
13              Delhi, India
14      Singapore, Singapore
15    Kuala Lumpur, Malaysia
16          Hyderabad, India
17          Kathmandu, Nepal
18      Melbourne, Australia
19         Sydney, Australia
20     Auckland, New Zealand
Name: city_name, dtype: object

In [29]:
gdf_classes = gdf.copy(["city_name"])
gdf_classes["class_2020"] = None
gdf_classes["class_2026"] = None

for i in tqdm(range(len(gdf))):
    curr_row = gdf.iloc[i]
    graph_2026 = ox.project_graph(curr_row["graph_2026"])
    graph_2020 = ox.project_graph(curr_row["graph_2020"])


    class_2026 = classify_city(graph_2026, model_path)
    class_2020 = classify_city(graph_2020, model_path)

    gdf_classes.at[curr_row.name, 'class_2026'] = class_2026
    gdf_classes.at[curr_row.name, 'class_2020'] = class_2020

    print()
    print()
    print(f"city: {curr_row['city_name']} 2020: {class_2020} 2026: {class_2026}")
    print()
    print()

Assigning edges to cells: 100%|██████████| 46008/46008 [00:08<00:00, 5150.33it/s]


Computing features for all subgraphs...


Processing subgraphs...: 100%|██████████| 1139/1139 [03:45<00:00,  5.05it/s]


Normalization...
Number of nodes: 18066
Number of features: 8
Подготовлено 1139 блоков с корректными признаками
Модель загружена


Assigning edges to cells: 100%|██████████| 31572/31572 [00:06<00:00, 4679.24it/s]


...



city: Chongqing, China 2020: Warped Parallel 2026: Loops & Lollipops




Assigning edges to cells: 100%|██████████| 20166/20166 [00:05<00:00, 3479.44it/s]


Computing features for all subgraphs...


 27%|██▋       | 3/11 [28:18<1:15:28, 566.05s/it]


KeyboardInterrupt: 

In [10]:
import pickle
import os
from tqdm import tqdm
import pandas as pd
import geopandas as gpd

# Параметры
FORCE_RERUN = False  # Установите True, если хотите перезаписать существующие файлы
os.makedirs("city_results", exist_ok=True)

# Основной цикл обработки
gdf_classes = gdf.copy(["city_name"])
gdf_classes["class_2020"] = None
gdf_classes["class_2026"] = None

all_results = []

# Загружаем список обработанных городов, если не принудительный запуск
processed_cities = set()
if not FORCE_RERUN:
    processed_file = "city_results/processed_cities.txt"
    if os.path.exists(processed_file):
        with open(processed_file, 'r') as f:
            processed_cities = set(f.read().splitlines())

for i in tqdm(range(len(gdf)), desc="Processing cities"):
    curr_row = gdf.iloc[i]
    if "Taipei, Taiwan" in curr_row.city_name:
        continue
    if "Osaka, Japan" in curr_row.city_name:
        continue
    if "Seoul, South Korea" in curr_row.city_name:
        continue
    
    safe_city_name = "".join(c for c in curr_row['city_name'] if c.isalnum() or c in (' ', '-', '_')).rstrip()
    base_filename = f"city_{safe_city_name}_{curr_row.name}"
    blocks_file = f"city_results/{base_filename}_blocks.pkl"
    city_info_file = f"city_results/{base_filename}_city_info.pkl"
    
    # Проверяем, нужно ли пропустить
    if not FORCE_RERUN:
        if os.path.exists(blocks_file) and os.path.exists(city_info_file):
            print(f"Skipping {curr_row['city_name']} - files already exist")
            # Восстанавливаем данные из существующих файлов для summary
            # try:
            #     # Пытаемся загрузить существующие данные для обновления сводки
            #     existing_blocks = pd.read_pickle(blocks_file)
            #     if 'predicted_class' in existing_blocks.columns:
            #         most_common_2020 = existing_blocks[existing_blocks['year'] == 2020]['predicted_class'].mode()[0] if 2020 in existing_blocks['year'].values else None
            #         most_common_2026 = existing_blocks[existing_blocks['year'] == 2026]['predicted_class'].mode()[0] if 2026 in existing_blocks['year'].values else None
            #         if most_common_2020 and most_common_2026:
            #             gdf_classes.at[curr_row.name, 'class_2026'] = most_common_2026
            #             gdf_classes.at[curr_row.name, 'class_2020'] = most_common_2020
            # except:
            #     pass
            continue
        
        if str(curr_row.name) in processed_cities:
            print(f"Skipping {curr_row['city_name']} - already in processed list")
            continue
    
    try:
        print(f"Processing {curr_row['city_name']}...")
        
        # Проектируем графы
        graph_2026 = ox.project_graph(curr_row["graph_2026"])
        graph_2020 = ox.project_graph(curr_row["graph_2020"])
        
        # Классифицируем для каждого года
        gdf_2026 = classify_city(graph_2026, model_path, year=2026)
        gdf_2020 = classify_city(graph_2020, model_path, year=2020)
        
        # Добавляем информацию о городе
        gdf_2026['city_name'] = curr_row['city_name']
        gdf_2020['city_name'] = curr_row['city_name']
        
        all_results.append(gdf_2026)
        all_results.append(gdf_2020)
        
        # Получаем самый популярный класс
        most_common_2026 = gdf_2026['predicted_class'].mode()[0]
        most_common_2020 = gdf_2020['predicted_class'].mode()[0]
        
        gdf_classes.at[curr_row.name, 'class_2026'] = most_common_2026
        gdf_classes.at[curr_row.name, 'class_2020'] = most_common_2020
        
        print(f"  city: {curr_row['city_name']} 2020: {most_common_2020} 2026: {most_common_2026}")
        print(f"  - Blocks 2020: {len(gdf_2020)}")
        print(f"  - Blocks 2026: {len(gdf_2026)}")
        
        # Сохраняем файлы
        combined_city_gdf = pd.concat([gdf_2020, gdf_2026], ignore_index=True)
        combined_city_gdf.to_pickle(blocks_file)
        
        if hasattr(curr_row, 'geometry') and curr_row.geometry is not None:
            curr_row_gdf = gpd.GeoDataFrame([curr_row], crs=gdf.crs)
        else:
            curr_row_gdf = pd.DataFrame([curr_row])
        
        curr_row_gdf.to_pickle(city_info_file)
        
        print(f"  ✓ Saved to {blocks_file} and {city_info_file}")
        
        # Записываем в список обработанных
        with open("city_results/processed_cities.txt", 'a') as f:
            f.write(f"{curr_row.name}\n")
            
    except Exception as e:
        print(f"Error processing city {curr_row['city_name']}: {e}")
        import traceback
        traceback.print_exc()
        continue

# # Сохраняем общие результаты
# if all_results:
#     combined_gdf = pd.concat(all_results, ignore_index=True)
#     combined_gdf.to_pickle("city_results/all_cities_blocks.pkl")
#     gdf_classes.to_pickle("city_results/city_summary.pkl")
#     gdf_classes.to_csv("city_results/city_summary.csv", index=False)
#     print(f"\n✓ Saved summary files to city_results/")
#     print(f"  Total blocks processed: {len(combined_gdf)}")
#     print(f"  Total cities processed: {len(gdf_classes[gdf_classes['class_2020'].notna()])}")
# else:
#     print("No new results were generated")

Processing cities:   0%|          | 0/21 [00:00<?, ?it/s]

Skipping Beijing, China - files already exist
Skipping Chengdu, China - files already exist
Skipping Chongqing, China - files already exist
Skipping Guangzhou, China - files already exist
Skipping Hangzhou, China - files already exist
Skipping Shanghai, China - files already exist
Skipping Shenzhen, China - files already exist
Skipping Wuhan, China - files already exist
Skipping Nanjing, China - files already exist
Processing Bangkok, Thailand...


Assigning edges to cells: 100%|██████████| 363988/363988 [01:10<00:00, 5153.74it/s]


Computing features for all subgraphs...


Processing subgraphs...: 100%|██████████| 1280/1280 [37:13<00:00,  1.74s/it]


...

Normalization...
Number of nodes: 95514
Number of features: 8
Подготовлено 1240 блоков с корректными признаками
Модель загружена


Assigning edges to cells: 100%|██████████| 544254/544254 [01:37<00:00, 5587.69it/s]


Computing features for all subgraphs...


Processing cities:  76%|███████▌  | 16/21 [12:32:23<3:55:07, 2821.45s/it]


KeyboardInterrupt: 